In [3]:
#!/usr/bin/env python3
"""
WALS Version: SV/VS vs GEN-N/N-GEN Correlation Analysis
Converted from Grambank analysis using exact feature mapping:
  - WALS 82A ↔ Grambank GB130 (SV/VS order)
  - WALS 86A ↔ Grambank GB065 (Genitive-Noun order)
  - WALS 87A ↔ Grambank GB193 (Adjective-Noun order)
  - WALS 24A ↔ Grambank GB431/GB433 (Head marking)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from tqdm import tqdm


def analyze_sv_gen_correlation(data, sample_num=None, verbose=False):
    """
    Analyze correlation between SV/VS order and GEN-N/N-GEN order.
    
    Hypothesis:
    - VS order correlates with N-GEN (PSSD-PSSR)
    - SV order correlates with GEN-N (PSSR-PSSD)
    
    WALS Features:
    - 82A: Order of Subject and Verb
    - 86A: Order of Genitive and Noun
    """
    
    data = data.copy()
    
    # WALS 82A: SV/VS order
    # '1' = SV (same as Grambank GB130)
    # '2' = VS (same as Grambank GB130)
    # '3' = No dominant order
    data['SV_Order'] = 'Unknown'
    data.loc[data['82A'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['82A'] == '2', 'SV_Order'] = 'VS'
    
    # WALS 86A: Genitive-Noun order
    # '1' = GEN-N (same as Grambank GB065)
    # '2' = N-GEN (same as Grambank GB065)
    # '3' = No dominant order
    data['GEN_Order'] = 'Unknown'
    data.loc[data['86A'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['86A'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # Filter valid data
    valid_data = data[(data['SV_Order'] != 'Unknown') & (data['GEN_Order'] != 'Unknown')]
    
    if len(valid_data) < 10:
        return None
    
    # Get SV and VS subsets
    sv_langs = valid_data[valid_data['SV_Order'] == 'SV']
    vs_langs = valid_data[valid_data['SV_Order'] == 'VS']
    
    # Count expected patterns
    # SV with GEN-N (expected)
    sv_gen_n = sum(sv_langs['GEN_Order'] == 'GEN-N')
    sv_total = len(sv_langs)
    
    # VS with N-GEN (expected)
    vs_n_gen = sum(vs_langs['GEN_Order'] == 'N-GEN')
    vs_total = len(vs_langs)
    
    # Calculate proportions
    sv_gen_n_prop = sv_gen_n / sv_total if sv_total > 0 else np.nan
    vs_n_gen_prop = vs_n_gen / vs_total if vs_total > 0 else np.nan
    
    # Binomial tests (null hypothesis: p = 0.5)
    sv_binom = binomtest(sv_gen_n, n=sv_total, p=0.5, alternative='greater') if sv_total >= 5 else None
    vs_binom = binomtest(vs_n_gen, n=vs_total, p=0.5, alternative='greater') if vs_total >= 5 else None
    
    sv_p = sv_binom.pvalue if sv_binom else np.nan
    vs_p = vs_binom.pvalue if vs_binom else np.nan
    
    if verbose:
        print(f"\nSample {sample_num}:")
        print(f"Total valid languages: {len(valid_data)}")
        print(f"SV languages: {sv_total}, SV→GEN-N: {sv_gen_n} ({sv_gen_n_prop:.1%}), p={sv_p:.4f}")
        print(f"VS languages: {vs_total}, VS→N-GEN: {vs_n_gen} ({vs_n_gen_prop:.1%}), p={vs_p:.4f}")
        print("-" * 70)
    
    return {
        'sample': sample_num,
        'n_languages': len(valid_data),
        'sv_total': sv_total,
        'vs_total': vs_total,
        'sv_gen_n': sv_gen_n,
        'vs_n_gen': vs_n_gen,
        'sv_gen_n_prop': sv_gen_n_prop,
        'vs_n_gen_prop': vs_n_gen_prop,
        'sv_p_value': sv_p,
        'vs_p_value': vs_p,
        'sv_significant': sv_p < 0.05 if not np.isnan(sv_p) else False,
        'vs_significant': vs_p < 0.05 if not np.isnan(vs_p) else False
    }


def analyze_sv_gen_head_marking(data, sample_num=None, verbose=False):
    """
    Same analysis for head-marking languages only
    
    WALS 24A: Locus of Marking in Possessive Noun Phrases
    Values: 1=Dependent-marking, 2=Head-marking, 3=Double-marking, 4=No marking
    
    We consider 2 (Head-marking) and 3 (Double-marking) as head-marking
    """
    
    if '24A' not in data.columns:
        return None
    
    # Head-marking includes both pure head-marking and double-marking
    head_marking = data[data['24A'].isin(['2', '3'])].copy()
    
    if len(head_marking) < 10:
        return None
    
    return analyze_sv_gen_correlation(head_marking, sample_num, verbose)


def create_stratified_sample(data, max_total=120, langs_per_area=20):
    """
    Create stratified sample following Hammarström & Donohue (2014) methodology.
    
    Parameters:
    - max_total: Maximum 120 languages total
    - langs_per_area: 20 language families per macroarea
    - All languages from different families
    """
    sampled_data = pd.DataFrame()
    
    # Get macroareas
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
        
        # Get unique families
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        # Sample up to langs_per_area families
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        # Sample one language per family
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                sampled_data = pd.concat([sampled_data, sampled_lang])
    
    # Ensure we don't exceed max_total
    if len(sampled_data) > max_total:
        sampled_data = sampled_data.sample(n=max_total)
    
    return sampled_data


def visualize_results(all_langs_results, head_marking_results):
    """Create comprehensive visualizations"""
    
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 14))
    
    # 1. SV → GEN-N proportion distribution
    ax = axes[0, 0]
    if len(all_df) > 0:
        sns.histplot(all_df['sv_gen_n_prop'].dropna() * 100, kde=True, bins=20, ax=ax, color='steelblue')
        mean_val = all_df['sv_gen_n_prop'].mean() * 100
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {mean_val:.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance (50%)')
        ax.set_xlabel('% SV languages with GEN-N', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('All Languages: SV → GEN-N Distribution (WALS)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 2. VS → N-GEN proportion distribution
    ax = axes[0, 1]
    if len(all_df) > 0:
        sns.histplot(all_df['vs_n_gen_prop'].dropna() * 100, kde=True, bins=20, ax=ax, color='coral')
        mean_val = all_df['vs_n_gen_prop'].mean() * 100
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {mean_val:.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance (50%)')
        ax.set_xlabel('% VS languages with N-GEN', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('All Languages: VS → N-GEN Distribution (WALS)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 3. P-value distribution - SV
    ax = axes[1, 0]
    if len(all_df) > 0:
        p_vals = all_df['sv_p_value'].dropna()
        sns.histplot(p_vals, bins=20, ax=ax, color='steelblue')
        ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
        sig_count = sum(p_vals < 0.05)
        ax.set_xlabel('P-value', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'SV P-values ({sig_count}/{len(p_vals)} significant)', 
                     fontsize=12, fontweight='bold')
        ax.legend()
    
    # 4. P-value distribution - VS
    ax = axes[1, 1]
    if len(all_df) > 0:
        p_vals = all_df['vs_p_value'].dropna()
        sns.histplot(p_vals, bins=20, ax=ax, color='coral')
        ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
        sig_count = sum(p_vals < 0.05)
        ax.set_xlabel('P-value', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'VS P-values ({sig_count}/{len(p_vals)} significant)', 
                     fontsize=12, fontweight='bold')
        ax.legend()
    
    # 5. Comparison: All vs Head-marking (SV)
    ax = axes[2, 0]
    if len(all_df) > 0:
        data_to_plot = [all_df['sv_gen_n_prop'].dropna() * 100]
        labels = ['All Languages']
        colors = ['steelblue']
        
        if len(hm_df) > 0:
            data_to_plot.append(hm_df['sv_gen_n_prop'].dropna() * 100)
            labels.append('Head-Marking')
            colors.append('coral')
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('% with GEN-N', fontsize=11)
        ax.set_title('SV → GEN-N Comparison', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    # 6. Comparison: All vs Head-marking (VS)
    ax = axes[2, 1]
    if len(all_df) > 0:
        data_to_plot = [all_df['vs_n_gen_prop'].dropna() * 100]
        labels = ['All Languages']
        colors = ['steelblue']
        
        if len(hm_df) > 0:
            data_to_plot.append(hm_df['vs_n_gen_prop'].dropna() * 100)
            labels.append('Head-Marking')
            colors.append('coral')
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('% with N-GEN', fontsize=11)
        ax.set_title('VS → N-GEN Comparison', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig('sv_vs_gen_correlation_stratified_WALS.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to: sv_vs_gen_correlation_stratified_WALS.png")
    plt.close()


def main():
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Load data
    print("Loading WALS data...")
    try:
        wals = pd.read_csv('wals_sane_format.csv')
        languages = pd.read_csv('wals_languages.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure WALS CSV files are in the current directory")
        print("Expected files:")
        print("  - wals_sane_format.csv")
        print("  - wals_languages.csv")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family']
            }
    
    wals['Macroarea'] = wals['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    wals['Family'] = wals['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns to string
    required_columns = ['82A', '86A', '87A', '24A']
    for col in required_columns:
        if col in wals.columns:
            wals[col] = wals[col].astype(str)
    
    print(f"Data loaded: {len(wals)} languages")
    
    # Run stratified analysis
    print("\n" + "="*70)
    print("WALS: SV/VS vs GEN-N/N-GEN CORRELATION ANALYSIS")
    print("Following Hammarström & Donohue (2014) methodology")
    print("="*70)
    print("\nFeature Mapping:")
    print("  WALS 82A (SV/VS) ↔ Grambank GB130")
    print("  WALS 86A (Genitive-Noun) ↔ Grambank GB065")
    print("  WALS 24A (Head-marking) ↔ Grambank GB431/GB433")
    print("\nHypotheses:")
    print("  H1: SV order correlates with GEN-N (PSSR-PSSD)")
    print("  H2: VS order correlates with N-GEN (PSSD-PSSR)")
    print("\nMethod: 300 stratified samples, max 120 languages each")
    print("        20 families per macroarea, binomial tests (p=0.5)")
    
    n_samples = 300
    all_langs_results = []
    head_marking_results = []
    
    print("\nProcessing samples...")
    for i in tqdm(range(n_samples), desc="Stratified sampling"):
        sample = create_stratified_sample(wals, max_total=120, langs_per_area=20)
        
        # All languages
        result_all = analyze_sv_gen_correlation(sample, sample_num=i+1, verbose=False)
        all_langs_results.append(result_all)
        
        # Head-marking
        result_hm = analyze_sv_gen_head_marking(sample, sample_num=i+1, verbose=False)
        head_marking_results.append(result_hm)
    
    # Summary statistics - All languages
    print("\n" + "="*70)
    print("RESULTS - ALL LANGUAGES")
    print("="*70)
    
    valid_all = [r for r in all_langs_results if r is not None]
    
    if len(valid_all) > 0:
        # SV → GEN-N
        sv_props = [r['sv_gen_n_prop'] * 100 for r in valid_all if not np.isnan(r['sv_gen_n_prop'])]
        sv_ps = [r['sv_p_value'] for r in valid_all if not np.isnan(r['sv_p_value'])]
        sv_sig = sum(1 for r in valid_all if r['sv_significant'])
        
        # VS → N-GEN
        vs_props = [r['vs_n_gen_prop'] * 100 for r in valid_all if not np.isnan(r['vs_n_gen_prop'])]
        vs_ps = [r['vs_p_value'] for r in valid_all if not np.isnan(r['vs_p_value'])]
        vs_sig = sum(1 for r in valid_all if r['vs_significant'])
        
        print(f"\nValid samples: {len(valid_all)}/{n_samples}")
        
        print(f"\nH1: SV → GEN-N (PSSR-PSSD)")
        print(f"  Mean: {np.mean(sv_props):.1f}% (±{np.std(sv_props):.1f}% SD)")
        print(f"  Median: {np.median(sv_props):.1f}%")
        print(f"  Range: [{np.min(sv_props):.1f}%, {np.max(sv_props):.1f}%]")
        print(f"  Significant samples: {sv_sig}/{len(sv_ps)} ({sv_sig/len(sv_ps):.1%})")
        
        if np.mean(sv_props) > 50:
            print(f"  Result: ✓ SUPPORTED (mean > 50%)")
        else:
            print(f"  Result: ✗ NOT SUPPORTED (mean < 50%)")
        
        print(f"\nH2: VS → N-GEN (PSSD-PSSR)")
        print(f"  Mean: {np.mean(vs_props):.1f}% (±{np.std(vs_props):.1f}% SD)")
        print(f"  Median: {np.median(vs_props):.1f}%")
        print(f"  Range: [{np.min(vs_props):.1f}%, {np.max(vs_props):.1f}%]")
        print(f"  Significant samples: {vs_sig}/{len(vs_ps)} ({vs_sig/len(vs_ps):.1%})")
        
        if np.mean(vs_props) > 50:
            print(f"  Result: ✓ SUPPORTED (mean > 50%)")
        else:
            print(f"  Result: ✗ NOT SUPPORTED (mean < 50%)")
    
    # Summary statistics - Head-marking
    print("\n" + "="*70)
    print("RESULTS - HEAD-MARKING LANGUAGES")
    print("="*70)
    
    valid_hm = [r for r in head_marking_results if r is not None]
    
    if len(valid_hm) > 0:
        # SV → GEN-N
        sv_props_hm = [r['sv_gen_n_prop'] * 100 for r in valid_hm if not np.isnan(r['sv_gen_n_prop'])]
        sv_sig_hm = sum(1 for r in valid_hm if r['sv_significant'])
        
        # VS → N-GEN
        vs_props_hm = [r['vs_n_gen_prop'] * 100 for r in valid_hm if not np.isnan(r['vs_n_gen_prop'])]
        vs_sig_hm = sum(1 for r in valid_hm if r['vs_significant'])
        
        print(f"\nValid samples: {len(valid_hm)}/{n_samples}")
        
        print(f"\nH1: SV → GEN-N (PSSR-PSSD)")
        print(f"  Mean: {np.mean(sv_props_hm):.1f}% (±{np.std(sv_props_hm):.1f}% SD)")
        print(f"  Significant samples: {sv_sig_hm}/{len(sv_props_hm)} ({sv_sig_hm/len(sv_props_hm):.1%})")
        
        print(f"\nH2: VS → N-GEN (PSSD-PSSR)")
        print(f"  Mean: {np.mean(vs_props_hm):.1f}% (±{np.std(vs_props_hm):.1f}% SD)")
        print(f"  Significant samples: {vs_sig_hm}/{len(vs_props_hm)} ({vs_sig_hm/len(vs_props_hm):.1%})")
        
        # Comparison
        print("\n" + "="*70)
        print("COMPARISON: ALL vs HEAD-MARKING")
        print("="*70)
        print(f"\nSV → GEN-N:")
        print(f"  All languages:  {np.mean(sv_props):.1f}%")
        print(f"  Head-marking:   {np.mean(sv_props_hm):.1f}%")
        print(f"  Difference:     {abs(np.mean(sv_props) - np.mean(sv_props_hm)):.1f} pp")
        
        print(f"\nVS → N-GEN:")
        print(f"  All languages:  {np.mean(vs_props):.1f}%")
        print(f"  Head-marking:   {np.mean(vs_props_hm):.1f}%")
        print(f"  Difference:     {abs(np.mean(vs_props) - np.mean(vs_props_hm)):.1f} pp")
    else:
        print("\nInsufficient head-marking samples for analysis")
    
    # Create visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_langs_results, head_marking_results)
    
    # Save results
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    all_df.to_csv('sv_vs_gen_all_languages_WALS.csv', index=False)
    print("Detailed results (all) saved to: sv_vs_gen_all_languages_WALS.csv")
    
    if len(valid_hm) > 0:
        hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
        hm_df.to_csv('sv_vs_gen_head_marking_WALS.csv', index=False)
        print("Detailed results (head-marking) saved to: sv_vs_gen_head_marking_WALS.csv")
    
    print("\n" + "="*70)
    print("WALS ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()



Loading WALS data...
Data loaded: 2659 languages

WALS: SV/VS vs GEN-N/N-GEN CORRELATION ANALYSIS
Following Hammarström & Donohue (2014) methodology

Feature Mapping:
  WALS 82A (SV/VS) ↔ Grambank GB130
  WALS 86A (Genitive-Noun) ↔ Grambank GB065
  WALS 24A (Head-marking) ↔ Grambank GB431/GB433

Hypotheses:
  H1: SV order correlates with GEN-N (PSSR-PSSD)
  H2: VS order correlates with N-GEN (PSSD-PSSR)

Method: 300 stratified samples, max 120 languages each
        20 families per macroarea, binomial tests (p=0.5)

Processing samples...


Stratified sampling: 100%|████████████████████| 300/300 [00:10<00:00, 30.00it/s]
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_95833/1394583997.py:243: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_95833/1394583997.py:265: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)



RESULTS - ALL LANGUAGES

Valid samples: 300/300

H1: SV → GEN-N (PSSR-PSSD)
  Mean: 82.3% (±3.9% SD)
  Median: 82.2%
  Range: [70.0%, 92.5%]
  Significant samples: 300/300 (100.0%)
  Result: ✓ SUPPORTED (mean > 50%)

H2: VS → N-GEN (PSSD-PSSR)
  Mean: 86.8% (±10.7% SD)
  Median: 87.5%
  Range: [50.0%, 100.0%]
  Significant samples: 173/291 (59.5%)
  Result: ✓ SUPPORTED (mean > 50%)

RESULTS - HEAD-MARKING LANGUAGES

Valid samples: 6/300

H1: SV → GEN-N (PSSR-PSSD)
  Mean: 89.5% (±8.7% SD)
  Significant samples: 4/6 (66.7%)

H2: VS → N-GEN (PSSD-PSSR)
  Mean: 94.4% (±12.4% SD)
  Significant samples: 0/6 (0.0%)

COMPARISON: ALL vs HEAD-MARKING

SV → GEN-N:
  All languages:  82.3%
  Head-marking:   89.5%
  Difference:     7.2 pp

VS → N-GEN:
  All languages:  86.8%
  Head-marking:   94.4%
  Difference:     7.6 pp

CREATING VISUALIZATIONS

Visualization saved to: sv_vs_gen_correlation_stratified_WALS.png
Detailed results (all) saved to: sv_vs_gen_all_languages_WALS.csv
Detailed results (h

In [4]:
#!/usr/bin/env python3
"""
WALS Version: Testing N-Raising Hypothesis for SV Languages
Compares two models for SV languages:
  Model 1 (Simple): SV → GEN-N
  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))

WALS Feature Mapping:
  - 82A ↔ GB130 (SV/VS order)
  - 86A ↔ GB065 (Genitive-Noun order)
  - 87A ↔ GB193 (Adjective-Noun order)
  - 24A ↔ GB431/GB433 (Head marking)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from tqdm import tqdm


def cohens_d(prop1, prop2, n1, n2):
    """
    Calculate Cohen's d for two proportions.
    
    Cohen's d = (p1 - p2) / pooled_SD
    where pooled_SD = sqrt(p_pooled * (1 - p_pooled))
    """
    # Pooled proportion
    p_pooled = (prop1 * n1 + prop2 * n2) / (n1 + n2)
    
    # Pooled standard deviation
    pooled_sd = np.sqrt(p_pooled * (1 - p_pooled))
    
    # Cohen's d
    if pooled_sd == 0:
        return np.nan
    
    d = (prop1 - prop2) / pooled_sd
    return d


def analyze_sv_patterns_with_adjectives(data, sample_num=None, verbose=False):
    """
    Compare two models for SV languages:
    
    Model 1 (Simple): SV → GEN-N
    Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
    
    Hypothesis: In SV languages with N-GEN, this results from N-raising,
    which should also produce N-Adj order.
    
    WALS Features:
    - 82A: Order of Subject and Verb (1=SV, 2=VS, 3=No dominant)
    - 86A: Order of Genitive and Noun (1=GEN-N, 2=N-GEN, 3=No dominant)
    - 87A: Order of Adjective and Noun (1=Adj-N, 2=N-Adj, 3=No dominant)
    """
    
    data = data.copy()
    
    # WALS 82A: SV/VS order
    data['SV_Order'] = 'Unknown'
    data.loc[data['82A'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['82A'] == '2', 'SV_Order'] = 'VS'
    
    # WALS 86A: Genitive order
    data['GEN_Order'] = 'Unknown'
    data.loc[data['86A'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['86A'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # WALS 87A: Adjective order
    data['Adj_Order'] = 'Unknown'
    data.loc[data['87A'] == '1', 'Adj_Order'] = 'Adj-N'
    data.loc[data['87A'] == '2', 'Adj_Order'] = 'N-Adj'
    
    # Filter to SV languages with all features
    sv_langs = data[(data['SV_Order'] == 'SV') & 
                    (data['GEN_Order'] != 'Unknown') & 
                    (data['Adj_Order'] != 'Unknown')].copy()
    
    if len(sv_langs) < 10:
        return None
    
    # MODEL 1: Simple - SV → GEN-N
    model1_match = sv_langs['GEN_Order'] == 'GEN-N'
    model1_count = sum(model1_match)
    model1_prop = model1_count / len(sv_langs)
    
    # MODEL 2: Complex - SV → (GEN-N OR (N-GEN AND N-Adj))
    # Pattern is "expected" if:
    # - GEN-N (regardless of adjective order), OR
    # - N-GEN AND N-Adj (the N-raising pattern)
    model2_match = ((sv_langs['GEN_Order'] == 'GEN-N') | 
                    ((sv_langs['GEN_Order'] == 'N-GEN') & (sv_langs['Adj_Order'] == 'N-Adj')))
    model2_count = sum(model2_match)
    model2_prop = model2_count / len(sv_langs)
    
    # Calculate improvement
    improvement = model2_prop - model1_prop
    improvement_pct = improvement * 100
    
    # Cohen's d
    d = cohens_d(model2_prop, model1_prop, len(sv_langs), len(sv_langs))
    
    # Binomial tests (null: p = 0.5)
    binom1 = binomtest(model1_count, n=len(sv_langs), p=0.5, alternative='greater')
    binom2 = binomtest(model2_count, n=len(sv_langs), p=0.5, alternative='greater')
    
    # Break down the N-GEN cases
    n_gen_langs = sv_langs[sv_langs['GEN_Order'] == 'N-GEN']
    n_gen_total = len(n_gen_langs)
    n_gen_with_n_adj = sum((n_gen_langs['Adj_Order'] == 'N-Adj'))
    n_gen_with_adj_n = sum((n_gen_langs['Adj_Order'] == 'Adj-N'))
    
    if verbose:
        print(f"\nSample {sample_num}:")
        print(f"Total SV languages: {len(sv_langs)}")
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Matches: {model1_count}/{len(sv_langs)} ({model1_prop:.1%})")
        print(f"  p-value: {binom1.pvalue:.4f}")
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Matches: {model2_count}/{len(sv_langs)} ({model2_prop:.1%})")
        print(f"  p-value: {binom2.pvalue:.4f}")
        print(f"\nImprovement: {improvement_pct:.1f} pp")
        print(f"Cohen's d: {d:.3f}")
        print(f"\nSV with N-GEN breakdown ({n_gen_total} languages):")
        print(f"  N-GEN + N-Adj (N-raising): {n_gen_with_n_adj} ({n_gen_with_n_adj/n_gen_total:.1%})")
        print(f"  N-GEN + Adj-N (unexpected): {n_gen_with_adj_n} ({n_gen_with_adj_n/n_gen_total:.1%})")
        print("-" * 70)
    
    return {
        'sample': sample_num,
        'n_sv': len(sv_langs),
        'model1_count': model1_count,
        'model1_prop': model1_prop,
        'model1_p': binom1.pvalue,
        'model2_count': model2_count,
        'model2_prop': model2_prop,
        'model2_p': binom2.pvalue,
        'improvement': improvement,
        'cohens_d': d,
        'n_gen_total': n_gen_total,
        'n_gen_n_adj': n_gen_with_n_adj,
        'n_gen_adj_n': n_gen_with_adj_n,
        'n_gen_n_adj_prop': n_gen_with_n_adj / n_gen_total if n_gen_total > 0 else np.nan
    }


def analyze_sv_patterns_head_marking(data, sample_num=None, verbose=False):
    """
    Same analysis for head-marking languages only
    
    WALS 24A: Locus of Marking in Possessive Noun Phrases
    Values: 1=Dependent-marking, 2=Head-marking, 3=Double-marking, 4=No marking
    """
    
    if '24A' not in data.columns:
        return None
    
    # Head-marking includes both pure head-marking and double-marking
    head_marking = data[data['24A'].isin(['2', '3'])].copy()
    
    if len(head_marking) < 10:
        return None
    
    return analyze_sv_patterns_with_adjectives(head_marking, sample_num, verbose)


def create_stratified_sample(data, max_total=120, langs_per_area=20):
    """Create stratified sample following Hammarström & Donohue (2014)"""
    sampled_data = pd.DataFrame()
    
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
        
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                sampled_data = pd.concat([sampled_data, sampled_lang])
    
    if len(sampled_data) > max_total:
        sampled_data = sampled_data.sample(n=max_total)
    
    return sampled_data


def visualize_results(all_langs_results, head_marking_results):
    """Create comprehensive visualizations"""
    
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1. Model comparison - All languages
    ax = axes[0, 0]
    if len(all_df) > 0:
        model1_props = all_df['model1_prop'].dropna() * 100
        model2_props = all_df['model2_prop'].dropna() * 100
        
        bp = ax.boxplot([model1_props, model2_props], 
                        labels=['Model 1:\nSV→GEN-N', 'Model 2:\nSV→(GEN-N OR\nN-GEN+N-Adj)'],
                        patch_artist=True)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightcoral')
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All Languages: Model Comparison (WALS)', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    # 2. Improvement distribution
    ax = axes[0, 1]
    if len(all_df) > 0:
        improvements = all_df['improvement'].dropna() * 100
        sns.histplot(improvements, kde=True, bins=20, ax=ax, color='green')
        ax.axvline(improvements.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {improvements.mean():.1f} pp')
        ax.axvline(0, color='black', linestyle=':', linewidth=2)
        ax.set_xlabel('Improvement (percentage points)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('Model 2 Improvement over Model 1', fontsize=12, fontweight='bold')
        ax.legend()
    
    # 3. Cohen's d distribution
    ax = axes[0, 2]
    if len(all_df) > 0:
        cohens_ds = all_df['cohens_d'].dropna()
        sns.histplot(cohens_ds, kde=True, bins=20, ax=ax, color='purple')
        ax.axvline(cohens_ds.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {cohens_ds.mean():.3f}')
        ax.axvline(0.2, color='gray', linestyle=':', alpha=0.5, label='Small')
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Medium')
        ax.axvline(0.8, color='gray', linestyle='-', alpha=0.5, label='Large')
        ax.set_xlabel("Cohen's d", fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title("Effect Size Distribution", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
    
    # 4. N-GEN breakdown
    ax = axes[1, 0]
    if len(all_df) > 0:
        n_adj_props = all_df['n_gen_n_adj_prop'].dropna() * 100
        sns.histplot(n_adj_props, kde=True, bins=20, ax=ax, color='orange')
        ax.axvline(n_adj_props.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {n_adj_props.mean():.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_xlabel('% of SV+N-GEN with N-Adj', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('N-raising Pattern in SV+N-GEN Languages', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 5. Head-marking comparison
    ax = axes[1, 1]
    if len(all_df) > 0:
        data_all = [all_df['model1_prop'] * 100, all_df['model2_prop'] * 100]
        labels_all = ['Model 1\n(All)', 'Model 2\n(All)']
        colors = ['lightblue', 'lightcoral']
        
        if len(hm_df) > 0:
            data_all.extend([hm_df['model1_prop'] * 100, hm_df['model2_prop'] * 100])
            labels_all.extend(['Model 1\n(Head-Mrk)', 'Model 2\n(Head-Mrk)'])
            colors.extend(['skyblue', 'salmon'])
        
        bp = ax.boxplot(data_all, labels=labels_all, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2)
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All vs Head-Marking Comparison', fontsize=12, fontweight='bold')
        ax.set_ylim(0, 100)
    
    # 6. Effect size comparison
    ax = axes[1, 2]
    if len(all_df) > 0:
        mean_d_all = all_df['cohens_d'].mean()
        std_d_all = all_df['cohens_d'].std()
        
        x = [0]
        means = [mean_d_all]
        stds = [std_d_all]
        labels = ['All Languages']
        colors_bar = ['purple']
        
        if len(hm_df) > 0:
            mean_d_hm = hm_df['cohens_d'].mean()
            std_d_hm = hm_df['cohens_d'].std()
            x.append(1)
            means.append(mean_d_hm)
            stds.append(std_d_hm)
            labels.append('Head-Marking')
            colors_bar.append('orange')
        
        bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors_bar, alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Cohen's d", fontsize=11)
        ax.set_title('Effect Size: Adding Adjective Order', fontsize=12, fontweight='bold')
        ax.axhline(0.2, color='gray', linestyle=':', alpha=0.5, label='Small')
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Medium')
        ax.axhline(0.8, color='gray', linestyle='-', alpha=0.5, label='Large')
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(means) + max(stds) + 0.2)
    
    plt.tight_layout()
    plt.savefig('sv_adjective_effect_analysis_WALS.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to: sv_adjective_effect_analysis_WALS.png")
    plt.close()


def main():
    # Set random seed
    np.random.seed(42)
    
    # Load data
    print("Loading WALS data...")
    try:
        wals = pd.read_csv('wals_sane_format.csv')
        languages = pd.read_csv('wals_languages.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure WALS CSV files are in the current directory")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family']
            }
    
    wals['Macroarea'] = wals['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    wals['Family'] = wals['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns
    required_columns = ['82A', '86A', '87A', '24A']
    for col in required_columns:
        if col in wals.columns:
            wals[col] = wals[col].astype(str)
    
    print(f"Data loaded: {len(wals)} languages")
    
    # Run analysis
    print("\n" + "="*70)
    print("WALS: TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES")
    print("="*70)
    print("\nFeature Mapping:")
    print("  WALS 82A (SV/VS) ↔ Grambank GB130")
    print("  WALS 86A (Genitive-Noun) ↔ Grambank GB065")
    print("  WALS 87A (Adjective-Noun) ↔ Grambank GB193")
    print("  WALS 24A (Head-marking) ↔ Grambank GB431/GB433")
    print("\nTheoretical Hypothesis:")
    print("  SV languages with N-GEN (unexpected pattern) result from N-raising,")
    print("  which should also produce N-Adj order")
    print("\nModel Comparison:")
    print("  Model 1 (Simple):  SV → GEN-N")
    print("  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
    print("\nEffect size: Cohen's d")
    
    n_samples = 300
    all_langs_results = []
    head_marking_results = []
    
    print("\nProcessing samples...")
    for i in tqdm(range(n_samples), desc="Stratified sampling"):
        sample = create_stratified_sample(wals, max_total=120, langs_per_area=20)
        
        result_all = analyze_sv_patterns_with_adjectives(sample, sample_num=i+1, verbose=False)
        all_langs_results.append(result_all)
        
        result_hm = analyze_sv_patterns_head_marking(sample, sample_num=i+1, verbose=False)
        head_marking_results.append(result_hm)
    
    # Summary statistics
    print("\n" + "="*70)
    print("RESULTS - ALL LANGUAGES")
    print("="*70)
    
    valid_all = [r for r in all_langs_results if r is not None]
    
    if len(valid_all) > 0:
        model1_props = [r['model1_prop'] * 100 for r in valid_all]
        model2_props = [r['model2_prop'] * 100 for r in valid_all]
        improvements = [r['improvement'] * 100 for r in valid_all]
        cohens_ds = [r['cohens_d'] for r in valid_all if not np.isnan(r['cohens_d'])]
        n_adj_props = [r['n_gen_n_adj_prop'] * 100 for r in valid_all if not np.isnan(r['n_gen_n_adj_prop'])]
        
        print(f"\nValid samples: {len(valid_all)}/{n_samples}")
        
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Mean: {np.mean(model1_props):.1f}% (±{np.std(model1_props):.1f}%)")
        print(f"  Range: [{np.min(model1_props):.1f}%, {np.max(model1_props):.1f}%]")
        
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Mean: {np.mean(model2_props):.1f}% (±{np.std(model2_props):.1f}%)")
        print(f"  Range: [{np.min(model2_props):.1f}%, {np.max(model2_props):.1f}%]")
        
        print(f"\nImprovement:")
        print(f"  Mean: {np.mean(improvements):.1f} percentage points (±{np.std(improvements):.1f})")
        print(f"  Range: [{np.min(improvements):.1f}, {np.max(improvements):.1f}] pp")
        
        print(f"\nEffect Size (Cohen's d):")
        print(f"  Mean: {np.mean(cohens_ds):.3f} (±{np.std(cohens_ds):.3f})")
        print(f"  Range: [{np.min(cohens_ds):.3f}, {np.max(cohens_ds):.3f}]")
        
        # Interpret Cohen's d
        mean_d = np.mean(cohens_ds)
        if mean_d < 0.2:
            interpretation = "negligible/small"
        elif mean_d < 0.5:
            interpretation = "small to medium"
        elif mean_d < 0.8:
            interpretation = "medium to large"
        else:
            interpretation = "large"
        print(f"  Interpretation: {interpretation} effect")
        
        print(f"\nN-raising Pattern (% of SV+N-GEN with N-Adj):")
        print(f"  Mean: {np.mean(n_adj_props):.1f}% (±{np.std(n_adj_props):.1f}%)")
        
        if np.mean(n_adj_props) > 50:
            print(f"  Result: ✓ N-raising hypothesis SUPPORTED")
        else:
            print(f"  Result: ✗ N-raising hypothesis NOT SUPPORTED")
    
    # Head-marking results
    print("\n" + "="*70)
    print("RESULTS - HEAD-MARKING LANGUAGES")
    print("="*70)
    
    valid_hm = [r for r in head_marking_results if r is not None]
    
    if len(valid_hm) > 0:
        model1_props_hm = [r['model1_prop'] * 100 for r in valid_hm]
        model2_props_hm = [r['model2_prop'] * 100 for r in valid_hm]
        improvements_hm = [r['improvement'] * 100 for r in valid_hm]
        cohens_ds_hm = [r['cohens_d'] for r in valid_hm if not np.isnan(r['cohens_d'])]
        
        print(f"\nValid samples: {len(valid_hm)}/{n_samples}")
        
        print(f"\nModel 1: {np.mean(model1_props_hm):.1f}% (±{np.std(model1_props_hm):.1f}%)")
        print(f"Model 2: {np.mean(model2_props_hm):.1f}% (±{np.std(model2_props_hm):.1f}%)")
        print(f"Improvement: {np.mean(improvements_hm):.1f} pp (±{np.std(improvements_hm):.1f})")
        print(f"Cohen's d: {np.mean(cohens_ds_hm):.3f} (±{np.std(cohens_ds_hm):.3f})")
        
        # Comparison
        print("\n" + "="*70)
        print("COMPARISON: ALL vs HEAD-MARKING")
        print("="*70)
        print(f"\nImprovement from adding adjective order:")
        print(f"  All languages:  {np.mean(improvements):.1f} pp")
        print(f"  Head-marking:   {np.mean(improvements_hm):.1f} pp")
        print(f"\nCohen's d:")
        print(f"  All languages:  {np.mean(cohens_ds):.3f}")
        print(f"  Head-marking:   {np.mean(cohens_ds_hm):.3f}")
    
    # Visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_langs_results, head_marking_results)
    
    # Save results
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    all_df.to_csv('sv_adjective_effect_all_WALS.csv', index=False)
    print("Detailed results (all) saved to: sv_adjective_effect_all_WALS.csv")
    
    if len(valid_hm) > 0:
        hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
        hm_df.to_csv('sv_adjective_effect_head_marking_WALS.csv', index=False)
        print("Detailed results (head-marking) saved to: sv_adjective_effect_head_marking_WALS.csv")
    
    print("\n" + "="*70)
    print("WALS ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()



Loading WALS data...
Data loaded: 2659 languages

WALS: TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES

Feature Mapping:
  WALS 82A (SV/VS) ↔ Grambank GB130
  WALS 86A (Genitive-Noun) ↔ Grambank GB065
  WALS 87A (Adjective-Noun) ↔ Grambank GB193
  WALS 24A (Head-marking) ↔ Grambank GB431/GB433

Theoretical Hypothesis:
  SV languages with N-GEN (unexpected pattern) result from N-raising,
  which should also produce N-Adj order

Model Comparison:
  Model 1 (Simple):  SV → GEN-N
  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))

Effect size: Cohen's d

Processing samples...


Stratified sampling: 100%|████████████████████| 300/300 [00:10<00:00, 29.27it/s]
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_95833/2307604203.py:216: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([model1_props, model2_props],
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_95833/2307604203.py:282: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_all, labels=labels_all, patch_artist=True)



RESULTS - ALL LANGUAGES

Valid samples: 300/300

Model 1 (Simple): SV → GEN-N
  Mean: 80.4% (±4.6%)
  Range: [63.3%, 93.8%]

Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
  Mean: 99.3% (±1.4%)
  Range: [94.1%, 100.0%]

Improvement:
  Mean: 18.9 percentage points (±4.4)
  Range: [6.2, 33.3] pp

Effect Size (Cohen's d):
  Mean: 0.624 (±0.094)
  Range: [0.349, 0.853]
  Interpretation: medium to large effect

N-raising Pattern (% of SV+N-GEN with N-Adj):
  Mean: 96.4% (±6.8%)
  Result: ✓ N-raising hypothesis SUPPORTED

RESULTS - HEAD-MARKING LANGUAGES

CREATING VISUALIZATIONS

Visualization saved to: sv_adjective_effect_analysis_WALS.png
Detailed results (all) saved to: sv_adjective_effect_all_WALS.csv

WALS ANALYSIS COMPLETE!
